# Project 98 — Local Data Pipeline Reviewer
## Pipeline Config → Quality Audit → Optimization Suggestions

**Stack:** LangChain · Ollama · Pydantic · Jupyter

In [ ]:
# !pip install -q langchain langchain-ollama pydantic pandas

## Step 1 — Pipeline Definitions

In [ ]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from pydantic import BaseModel, Field
import json, pandas as pd

llm = ChatOllama(model="qwen3:8b", temperature=0.1)

pipelines = [
    {
        "name": "user_events_etl",
        "stages": [
            {"name": "ingest", "type": "kafka_consumer", "config": {"topic": "user_events", "batch_size": 1000}},
            {"name": "validate", "type": "schema_check", "config": {"schema": "user_event_v2", "strict": False}},
            {"name": "transform", "type": "python", "config": {"script": "transform_events.py", "timeout": 300}},
            {"name": "deduplicate", "type": "window_dedup", "config": {"window": "1h", "key": "event_id"}},
            {"name": "load", "type": "postgres_insert", "config": {"table": "events", "batch_size": 500}},
        ],
        "schedule": "*/5 * * * *",
        "owner": "data-team",
    },
    {
        "name": "daily_aggregates",
        "stages": [
            {"name": "extract", "type": "sql_query", "config": {"query": "SELECT * FROM events WHERE date = CURRENT_DATE - 1"}},
            {"name": "aggregate", "type": "pandas", "config": {"groupby": ["user_id", "event_type"], "agg": "count"}},
            {"name": "enrich", "type": "api_call", "config": {"endpoint": "/users/batch", "batch_size": 100}},
            {"name": "load", "type": "bigquery_write", "config": {"table": "analytics.daily_summary", "mode": "append"}},
        ],
        "schedule": "0 2 * * *",
        "owner": "analytics-team",
    },
    {
        "name": "ml_feature_pipeline",
        "stages": [
            {"name": "source", "type": "s3_read", "config": {"bucket": "raw-data", "prefix": "features/"}},
            {"name": "clean", "type": "spark", "config": {"script": "clean_features.py", "memory": "8g"}},
            {"name": "compute", "type": "spark", "config": {"script": "compute_features.py", "memory": "16g"}},
            {"name": "validate", "type": "great_expectations", "config": {"suite": "feature_suite"}},
            {"name": "publish", "type": "feature_store_write", "config": {"store": "feast", "entity": "user_features"}},
        ],
        "schedule": "0 4 * * *",
        "owner": "ml-team",
    },
]
print(f"Pipelines to review: {len(pipelines)}")
for p in pipelines:
    print(f"  {p['name']}: {len(p['stages'])} stages, schedule={p['schedule']}")

## Step 2 — Pipeline Quality Audit

In [ ]:
class StageIssue(BaseModel):
    stage_name: str
    issue_type: str = Field(description="performance, reliability, security, data_quality")
    severity: str = Field(description="critical, high, medium, low")
    description: str
    recommendation: str

class PipelineAudit(BaseModel):
    pipeline_name: str
    health_score: float = Field(ge=0, le=10)
    issues: list[StageIssue]
    missing_stages: list[str]
    redundant_stages: list[str]
    data_quality_gaps: list[str]
    scalability_concerns: list[str]
    estimated_reliability: float = Field(ge=0, le=1)

auditor = llm.with_structured_output(PipelineAudit)

audits = []
for pipeline in pipelines:
    audit = auditor.invoke(
        f"Audit this data pipeline for quality and reliability:\n\n"
        f"{json.dumps(pipeline, indent=2)}"
    )
    audits.append(audit)
    print(f"\n{audit.pipeline_name}: score={audit.health_score}/10 reliability={audit.estimated_reliability:.0%}")
    print(f"  Issues: {len(audit.issues)}")
    for issue in audit.issues[:3]:
        print(f"    [{issue.severity}] {issue.stage_name}: {issue.description[:60]}")
    if audit.missing_stages:
        print(f"  Missing: {audit.missing_stages}")

## Step 3 — Optimization Suggestions

In [ ]:
optimize_prompt = ChatPromptTemplate.from_messages([
    ("system", "Suggest specific optimizations for this pipeline. "
     "Include: performance improvements, cost reductions, reliability enhancements. "
     "Provide concrete config changes."),
    ("human", "Pipeline: {name}\nStages: {stages}\nIssues: {issues}\n\nOptimizations:")
])
optimize_chain = optimize_prompt | llm | StrOutputParser()

for pipeline, audit in zip(pipelines, audits):
    optimizations = optimize_chain.invoke({
        "name": pipeline["name"],
        "stages": json.dumps(pipeline["stages"]),
        "issues": json.dumps([i.model_dump() for i in audit.issues]),
    })
    print(f"\n{'='*50}")
    print(f"OPTIMIZATIONS: {pipeline['name']}")
    print(optimizations[:400])

## Step 4 — Pipeline Health Dashboard

In [ ]:
rows = []
for audit in audits:
    rows.append({
        "pipeline": audit.pipeline_name,
        "health": audit.health_score,
        "reliability": f"{audit.estimated_reliability:.0%}",
        "issues": len(audit.issues),
        "critical": sum(1 for i in audit.issues if i.severity == "critical"),
        "missing_stages": len(audit.missing_stages),
    })

df = pd.DataFrame(rows)
print("PIPELINE HEALTH DASHBOARD")
print("=" * 60)
print(df.to_string(index=False))

print(f"\nOverall health: {df['health'].mean():.1f}/10")
print(f"Critical issues: {df['critical'].sum()}")

# Issue breakdown
all_issues = []
for audit in audits:
    for issue in audit.issues:
        all_issues.append({
            "pipeline": audit.pipeline_name,
            "type": issue.issue_type,
            "severity": issue.severity,
        })
idf = pd.DataFrame(all_issues)
if len(idf) > 0:
    print(f"\nIssues by type: {idf['issue_type'].value_counts().to_dict()}")
    print(f"Issues by severity: {idf['severity'].value_counts().to_dict()}")

## What You Learned
- **Pipeline configuration auditing** with structured analysis
- **Multi-dimension quality scoring** (performance, reliability, security)
- **Optimization recommendations** with concrete changes
- **Health dashboard** for pipeline monitoring